In [2]:
#import nflfastpy as nfl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as plticker
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
import seaborn as seabornInstance
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm



data = {
    "schedule_date": ["2025-09-04", "2025-09-05", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-07", "2025-09-08"],
    "schedule_season": [2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025, 2025],
    "schedule_week": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    "team_home_id": ["PHI", "LAC", "ATL", "CLE", "IND", "NE", "NO", "NYJ", "WAS", "JAX", "DEN", "SEA", "GB", "LAR", "BUF"],
    "team_away_id": ["DAL", "KC", "TB", "CIN", "MIA", "LV", "ARI", "PIT", "NYG", "CAR", "TEN", "SF", "DET", "HOU", "BAL"]
}

base_data = pd.DataFrame(data)
#Need to update this to latest csv file generated based on last weeks game results
# csv_file = 'wk1_forecast.csv'
# base_data = pd.read_csv(csv_file)

#Pull PBP data from repo
YEARS = [*range(2000, 2025)]

epa_data = pd.DataFrame()

for year in YEARS:  
    i_data = pd.read_csv('epa_pbp_2000-2024/play_by_play_' + str(year) + '.csv')

    epa_data = pd.concat([epa_data, i_data], ignore_index=True, sort=True)

/var/folders/pj/ql6w6yr508zfwm_nzbbz14rw0000gn/T/ipykernel_58585/602508994.py:36: DtypeWarning: Columns (37,45,179,180,182,183,189,190,193,194,197,198,203,204,205,206,218,219,220,248,249,253,254,255,260,262,263,283,284) have mixed types. Specify dtype option on import or set low_memory=False.
  i_data = pd.read_csv('epa_pbp_2000-2024/play_by_play_' + str(year) + '.csv')
/var/folders/pj/ql6w6yr508zfwm_nzbbz14rw0000gn/T/ipykernel_58585/602508994.py:36: DtypeWarning: Columns (37,45,179,180,182,183,189,190,193,194,197,198,203,204,205,206,209,210,213,214,218,219,220,248,249,253,254,255,260,262,263,266,267,268,269,283,284,301) have mixed types. Specify dtype option on import or set low_memory=False.
  i_data = pd.read_csv('epa_pbp_2000-2024/play_by_play_' + str(year) + '.csv')
/var/folders/pj/ql6w6yr508zfwm_nzbbz14rw0000gn/T/ipykernel_58585/602508994.py:36: DtypeWarning: Columns (37,45,179,180,182,183,189,190,193,194,197,198,203,204,205,206,209,210,218,219,220,222,224,226,248,249,253,254,255

In [3]:
def dynamic_window_ewma(x):
    """
    Calculate rolling exponentially weighted EPA with a dynamic window size
    """
    values = np.zeros(len(x))
    for i, (_, row) in enumerate(x.iterrows()):
        epa = x.epa_shifted[:i+1]
        if row.week > 10:
            values[i] = epa.ewm(min_periods=1, span=row.week).mean().values[-1]
        else:
            values[i] = epa.ewm(min_periods=1, span=10).mean().values[-1]
            
    return pd.Series(values, index=x.index)


rushing_offense_epa = epa_data.loc[epa_data['rush_attempt'] == 1, :]\
.groupby(['posteam', 'season', 'week'], as_index=False)['epa'].mean()

rushing_defense_epa = epa_data.loc[epa_data['rush_attempt'] == 1, :]\
.groupby(['defteam', 'season', 'week'], as_index=False)['epa'].mean()

passing_offense_epa = epa_data.loc[epa_data['pass_attempt'] == 1, :]\
.groupby(['posteam', 'season', 'week'], as_index=False)['epa'].mean()

passing_defense_epa = epa_data.loc[epa_data['pass_attempt'] == 1, :]\
.groupby(['defteam', 'season', 'week'], as_index=False)['epa'].mean()

In [4]:
orig_epa_df_dict = {'rushing_offense_epa':rushing_offense_epa, 'rushing_defense_epa': rushing_defense_epa, 'passing_offense_epa': passing_offense_epa, 'passing_defense_epa': passing_defense_epa}

forecast_epa_df_dict = {}

for key, df in orig_epa_df_dict.items():
    current_season = df['season'].max()
    forecast_wk = df.loc[df['season'] == current_season, 'week'].max() + 1

    # Team key differs for offense vs defense
    team_col = 'posteam' if 'offense' in key else 'defteam'
    df_forecast_name = key + '_forecast'

    # Get max completed week per team this season
    team_completed_wk = (
        df.loc[df['season'] == current_season]
        .groupby(team_col)['week']
        .max()
    )

    # Keep all rows except future weeks
    forecast_df = df.copy()

    # Add one forecast row per team
    forecast_rows = pd.DataFrame({
        team_col: team_completed_wk.index,
        'season': current_season,
        'week': forecast_wk
    })
    forecast_df = pd.concat([forecast_df, forecast_rows], ignore_index=True)

    # Add shifted EPA
    forecast_df['epa_shifted'] = (
        forecast_df.groupby(team_col)['epa'].shift()
    )

    # Apply dynamic EWMA per team
    forecast_df['ewma_dynamic_window'] = (
        forecast_df.groupby(team_col, group_keys=False)
        .apply(dynamic_window_ewma)
        .reset_index(drop=True)
    )

    # Save result
    forecast_epa_df_dict[df_forecast_name] = forecast_df

/var/folders/pj/ql6w6yr508zfwm_nzbbz14rw0000gn/T/ipykernel_58585/2124542770.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(dynamic_window_ewma)
/var/folders/pj/ql6w6yr508zfwm_nzbbz14rw0000gn/T/ipykernel_58585/2124542770.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(dynamic_window_ewma)
/var/folders/pj/ql6w6yr508zfwm_nzbbz14rw0000gn/T/ipykernel_58585/2124542770.py:39: FutureWarning: DataFram

In [6]:
offense_epa = forecast_epa_df_dict['rushing_offense_epa_forecast'].merge(forecast_epa_df_dict['passing_offense_epa_forecast'], on=['posteam', 'season', 'week'], suffixes=('_rushing', '_passing'))\
.rename(columns={'posteam': 'team'})

defense_epa = forecast_epa_df_dict['rushing_defense_epa_forecast'].merge(forecast_epa_df_dict['passing_defense_epa_forecast'], on=['defteam', 'season', 'week'], suffixes=('_rushing', '_passing'))\
.rename(columns={'defteam': 'team'})

epa = offense_epa.merge(defense_epa, on=['team', 'season', 'week'], suffixes=('_offense', '_defense'))

#update team names before joining
epa['team'] = epa['team'].replace('LA','LAR')

epa_basic = epa[['team','season','week','ewma_dynamic_window_rushing_offense','ewma_dynamic_window_passing_offense','ewma_dynamic_window_rushing_defense','ewma_dynamic_window_passing_defense']]

df_home_epa = base_data.merge(epa_basic, how='left', left_on=['team_home_id', 'schedule_season', 'schedule_week'], right_on=['team','season','week'])\
.rename(columns={'ewma_dynamic_window_rushing_offense':'ewma_dynamic_window_rushing_offense_home','ewma_dynamic_window_passing_offense':'ewma_dynamic_window_passing_offense_home',
                 'ewma_dynamic_window_rushing_defense':'ewma_dynamic_window_rushing_defense_home','ewma_dynamic_window_passing_defense':'ewma_dynamic_window_passing_defense_home','team':'home_team_epa'})

df_full_epa = df_home_epa.merge(epa_basic, how='left', left_on=['team_away_id', 'schedule_season', 'schedule_week'], right_on=['team','season','week'])\
.rename(columns={'ewma_dynamic_window_rushing_offense':'ewma_dynamic_window_rushing_offense_away','ewma_dynamic_window_passing_offense':'ewma_dynamic_window_passing_offense_away',
                 'ewma_dynamic_window_rushing_defense':'ewma_dynamic_window_rushing_defense_away','ewma_dynamic_window_passing_defense':'ewma_dynamic_window_passing_defense_away','team':'away_team_epa'})

In [8]:
df_full_epa.to_csv("epa_csv")